In [1]:
import pandas as pd                              

Data from https://www.fec.gov/data/browse-data/?tab=bulk-data, specifically "contributions by individials" for 2025-2026. This notebook loads the raw data file and adds clean versions of some columns, ultimately outputting two files: one with the format of each string and one with the format classes and their counts.

In [2]:
def select_columns(data_file: pd.DataFrame, header_file: pd.DataFrame, columns_to_select: list):
    """
    Selects specific columns from the FEC contribution data, and add headers.
    """
    column_names = header_file.columns.astype(str).tolist()
    data_file.columns = column_names
    filtered_df = data_file[columns_to_select]
    return filtered_df

In [3]:
def clean_names(df):
    """
    Make clean names, and FIRST_NAME and LAST_NAME columns from NAME column.
    Keeps only:
    - LAST_NAME: part before comma
    - FIRST_NAME: first word after comma
    Also rewrites NAME as "FIRST_NAME LAST_NAME" in CLEAN_NAME.
    """
    no_comma = ~df["NAME"].fillna("").astype(str).str.contains(",")
    df.drop(index=df[no_comma].index, inplace=True)

    name_series = df["NAME"].fillna("").astype(str).str.strip()

    # For values like "BAKER, BRIAN H.":
    # LAST_NAME = "BAKER", FIRST_NAME = "BRIAN"
    name_parts = name_series.str.split(",", n=1, expand=True)

    last_name = name_parts[0].fillna("").str.strip()
    right_side = name_parts[1].fillna("").str.strip()
    # Separate the first word after the comma as FIRST_NAME, ignoring middle names or initials.
    first_name = right_side.str.split(r"\s+", n=1, expand=True)[0].fillna("").str.strip()

    df["FIRST_NAME"] = first_name
    df["LAST_NAME"] = last_name
    df["CLEAN_NAME"] = (df["FIRST_NAME"] + " " + df["LAST_NAME"]).str.replace(r"\s+", " ", regex=True).str.strip()

    # Null CLEAN_NAME when the cleaned name still contains a comma or either component contains spaces.
    invalid_format = (
        df["CLEAN_NAME"].astype(str).str.contains(",")
        | df[["FIRST_NAME", "LAST_NAME"]].astype(str).apply(lambda x: x.str.contains(r"\s+")).any(axis=1)
    )
    df.loc[invalid_format, "CLEAN_NAME"] = pd.NA

    # Null CLEAN_NAME when it contains any improper-name character.
    improper_name_pattern = r"[+/@.&0-9#$%^*=|~<>(){}\[\]_\\;:?!]"
    improper_name = df["CLEAN_NAME"].astype(str).str.contains(improper_name_pattern, na=False)
    df.loc[improper_name, "CLEAN_NAME"] = pd.NA

    # Null CLEAN_NAME when it contains an organization keyword.
    org_keywords = r"COMMITTEE|COUNCIL|ASSOCIATION|FUND|PAC|CORP|LLC|INC|LTD|TRUST|FOUNDATION"
    organization_name = df["CLEAN_NAME"].astype(str).str.contains(org_keywords, case=False, regex=True, na=False)
    df.loc[organization_name, "CLEAN_NAME"] = pd.NA

    # Keep name components consistent with CLEAN_NAME.
    null_clean_name = df["CLEAN_NAME"].isna()
    df.loc[null_clean_name, ["FIRST_NAME", "LAST_NAME"]] = pd.NA

In [4]:
selected_columns = ["NAME", "CITY", "ZIP_CODE", "EMPLOYER", "OCCUPATION"]
# FEC contribution files are pipe-delimited and have no header row.
df = pd.read_csv("Raw Data/itcont.txt", sep='|', header=None, low_memory=False)
header = pd.read_csv("Raw Data/indiv_header_file.csv")
df = select_columns(df, header, selected_columns)
df.drop_duplicates(inplace=True)
clean_names(df)
df.to_csv("Processed Data/cleaned_data.csv", index=False)

In [5]:
def name_map(name: str) -> str:
    """
    Map name to format string where only spaces are preserved.
    """
    mapped_name = ''
    for ch in name:
        if ch == " ":
            mapped_name += ch
        else:
            mapped_name += 'x'
    return mapped_name

In [6]:
def city_map(city: str) -> str:
    """
    Map city to format string where spaces are preserved.
    """
    mapped_city = ''
    for ch in city:
        if ch == " ":
            mapped_city += ch
        else:
            mapped_city += 'x'
    return mapped_city


In [7]:
def zip_map(name: str) -> str:
    """
    Map zip code to format string where length is preserved.
    """
    mapped_zip = ''
    for ch in name:
        mapped_zip += 'x'
    return mapped_zip

In [8]:
def employer_map(name: str) -> str:
    """
    Map employer to format string where only spaces are preserved.
    """
    mapped_name = ''
    for ch in name:
        if ch == " ":
            mapped_name += ch
        else:
            mapped_name += 'x'
    return mapped_name

In [9]:
def occupation_map(name: str) -> str:
    """
    Map occupation to format string where only spaces are preserved.
    """
    mapped_name = ''
    for ch in name:
        if ch == " ":
            mapped_name += ch
        else:
            mapped_name += 'x'
    return mapped_name

In [10]:
col_map_fns = {
    "NAME": name_map,
    "CLEAN_NAME": name_map,
    "FIRST_NAME": name_map,
    "LAST_NAME": name_map,
    "CITY": city_map,
    "ZIP_CODE": zip_map,
    "EMPLOYER": employer_map,
    "OCCUPATION": occupation_map,
}

df_mapped = df.copy()
for col, fn in col_map_fns.items():
    if col in df_mapped.columns:
        df_mapped[col] = df_mapped[col].fillna("").astype(str).map(fn)

df_mapped.to_csv("Processed Data/formats.csv", index=False)